In [ ]:
import os
import anthropic
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env iaectory

In [ ]:
client = anthropic.Anthropic(
    base_url=os.environ.get("ANTHROPIC_BASE_URL"),
    api_key=os.environ.get("ANTHROPIC_AUTH_TOKEN"),
    
)

MODEL = os.environ.get("ANTHROPIC_SMALL_FAST_MODEL")

print(f"Using model: {MODEL}")
print(f"Base URL:    {os.environ.get('ANTHROPIC_BASE_URL')}")

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user",
                    "content": text,
                   }
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant",
                         "content": text,
                   }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=0.0, stop_sequences=[]):
    param = {
        "model": MODEL,
        "max_tokens": 1024,
        "messages" : messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        param["system"] = system
    
    message = client.messages.create(**param)
    return message.content[0].text    

In [ ]:
import json

In [ ]:
def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.
    
    Example output:
    ```json
    [
        {
            "task": "Description of task",
        },
    ...additional
    ]
    ```
    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code
    
    Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)   


In [ ]:
dataset = generate_dataset()

In [ ]:
dataset

In [ ]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
from statistics import mean

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Solve the following task:

    {test_case["task"]}
    """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)    

In [ ]:
results

In [ ]:
print(json.dumps(results, indent=2))

In [ ]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.
    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>
    
    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10
    
    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))